## B2 American Airlines Project

## Problem Statement
Once a flight is already airborne, arrival time estimates may still change due to en-route conditions or operational adjustments. These post-takeoff changes can create downstream inefficiencies for airline operations and passenger connections.

## The central question of this project is:
Given a flight is already airborne, can we predict whether its arrival time will change, and if so, by how many minutes, using only information available at or shortly after takeoff?

## Objective and Expected Outcome
The objective of this project is to develop a model that predicts the change in arrival time (Δ), measured in minutes, relative to the arrival estimate available at takeoff.

## Target Definition
## Δ = Final Arrival Time − Arrival Estimate at Takeoff
The model outputs a predicted Δ in minutes, where positive values indicate the flight arrived later than expected and negative values indicate the flight arrived earlier than expected.

## Evaluation Metric
Model performance will be evaluated using Mean Absolute Error (MAE), measured in minutes. MAE is chosen for its interpretability and operational relevance, as average deviation in minutes is more meaningful for airline decision-making than squared error metrics.

## Scope Clarification: What We Are Not Predicting
This project intentionally narrows its scope. We are not attempting to predict overall flight delays from scheduling time, optimize departure schedules, or explain the root causes of delays across the full flight lifecycle.

Instead, the focus is specifically on post-takeoff arrival time changes, isolating a decision-critical window where updated predictions can meaningfully support operational planning.


# Loading the Data (Cleaning & Observational Analysis)

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

seq_spoil = pd.read_csv(
    "../data/raw/sequence_spoilage.csv",
    sep="|", #the separator is a pipe character
    engine="python" #
)
assign_history = pd.read_csv(
    "../data/raw/Assignment_History.csv",
    sep=",", #the separator is a pipe character
    engine="python" #specify the engine to avoid warnings
)

# Drop the first column which is unnamed for both dataframes
seq_spoil.drop(columns=[seq_spoil.columns[0]], inplace=True) 
seq_spoil.head()

assign_history.drop(columns=[assign_history.columns[0]], inplace=True)
assign_history.head()

#57,067 entries in the assign_history dataframe
##assign_history.info()

#9025 entries in the seq_spoil dataframe
seq_spoil.info()

seq_spoil.isna().sum() #contains null values in the "TOTAL_BLOCKED_HRS" column & "TOTAL_SPOILED_HRS" column

assign_history.isna().sum() #contains a multitude of missing values across various columns



<class 'pandas.DataFrame'>
RangeIndex: 9025 entries, 0 to 9024
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SEQ_NBR              9025 non-null   int64  
 1   SEQ_SCHD_START_DT    9025 non-null   str    
 2   FLEET                9025 non-null   int64  
 3   BASE                 9025 non-null   str    
 4   DIVISION             9025 non-null   str    
 5   SPOILAGE             9025 non-null   str    
 6   TOTAL_BLOCKED_HRS    4420 non-null   float64
 7   TOTAL_SPOILED_HRS    2719 non-null   float64
 8   SEQ_CAL_DAYS         9025 non-null   int64  
 9   SEQ_DUTY_DAYS        9025 non-null   int64  
 10  SEQ_TTL_FLTTIME      9025 non-null   int64  
 11  MIN_FLYTIME_PER_LEG  9025 non-null   int64  
 12  MAX_LEGS_PER_DAY     9025 non-null   int64  
 13  SEQ_TTL_LEGS         9025 non-null   int64  
 14  MORETHAN2_321_LEGS   9025 non-null   int64  
 15  IN_SEQ_DHD           9025 non-null   int64  
 16 

CONTRCT_YR                   0
CONTRCT_MONTH                0
FLEET_CD                     0
SEQ_NBR                      0
SEQ_ORIGIN_DT                0
SEQ_BASE                  1446
IS_BLOCK_OE_IND           1446
SEQ_OE_SNPSHT_ID         10617
SEQ_ID                    1446
SEQ_SOURCE_CD             1446
SEQ_ROW_EFF_TMS           1446
SEQ_ROW_EXPIRY_TMS        1446
TRIP_ID                   1451
TRIP_SOURCE_CD           28962
TRIP_ROW_EFF_TMS          1451
TRIP_ROW_EXPIRY_TMS       1451
SEQ_POSITN_CD             1446
FOS_SEQ_STATUS_TXT        7830
STATUS_TXT                1446
CKP_EMP_NBR              17082
CKP_BASE                 17082
CKP_ROLE_TYPE_DESC       17807
STUDENT_EMP_NBR          40197
STUDENT_POSITN_CD        40197
STUDENT_TRNG_TYPE        40302
STUDENT_BASE             40197
IS_STUDENT_OE_STUDENT    40197
STUDENT_OE_START_DT      40669
STUDENT_OE_END_DT        40669
TRIP_QLA_MSG_JSON_TXT     6155
TRIP_COORD_CMNT_TXT      33492
SEQ_DUTY_PERIOD_CT        1446
SEQ_CALN

## Data Inventory & Target Feasibility 
Can we compute (Delta) for enough flight?

Tasks:
- Identify keys: sequence, sequence number, sequence leg (ETA), final arrival time
- create list of usable flights (sample size)

Basic Target Distribution Plot (histogram of (delta))

Exit Criteria:

At least a meaningful sample (whatever your project needs) where (delta) is computable.


## Define Unit of Observation + Build the "As-of-Snapshot"
One row per flight at the prediction moment

Tasks:
- If you have multiple ETA updates per flight: select the latest ETA at or before as-of
- Building a snapshot table: where each row is a flight as-of
    - Table shold include:
        ID, date, Origin, Destination
        As-of Timestamp
        ETA at as-of
        final arrival
        target(delta)
        candidate features

Exit Criteria 

No duplicate flight_id rows in snapshot

(delta) is computed cleanly

 
